In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Cargar el dataset MNIST (imágenes de 28x28 píxeles de dígitos a mano)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 2. Preprocesamiento
# Normalizar los valores de píxeles de 0-255 a 0-1
x_train, x_test = x_train / 255.0, x_test / 255.0
# Añadir una dimensión extra para el canal de color (Blanco y negro = 1 canal)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 3. Construir la arquitectura CNN
model = models.Sequential([
    # Capa de convolución: detecta características (bordes, curvas)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)), # Reduce la dimensionalidad
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Aplanar los datos para la parte densa (clasificación)
    layers.Flatten(),
    layers.Dropout(0.5), # Apaga neuronas al azar para evitar overfitting (mencionado en tu imagen)
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax') # 10 salidas (dígitos 0-9)
])

# 4. Compilar y Entrenar
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("Entrenando modelo...")
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# 5. Guardar el modelo
model.save('modelo_digitos.keras')
print("¡Modelo guardado como 'modelo_digitos.keras'!")

c:\Users\franc\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando modelo...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - accuracy: 0.9449 - loss: 0.1759 - val_accuracy: 0.9836 - val_loss: 0.0477
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9794 - loss: 0.0665 - val_accuracy: 0.9896 - val_loss: 0.0323
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9842 - loss: 0.0497 - val_accuracy: 0.9915 - val_loss: 0.0260
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 41s 11ms/step - accuracy: 0.9867 - loss: 0.0403 - val_accuracy: 0.9921 - val_loss: 0.0256
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.9882 - loss: 0.0348 - val_accuracy: 0.9927 - val_loss: 0.0228
¡Modelo guardado como 'modelo_digitos.keras'!


In [3]:
import cv2
import numpy as np
import tensorflow as tf

# Cargar el modelo entrenado
model = tf.keras.models.load_model('modelo_digitos.keras')

drawing = False # Estado del mouse
ix, iy = -1, -1

# Función para dibujar con el mouse
def draw_circle(event, x, y, flags, param):
    global ix, iy, drawing, img
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y
    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            # Dibujamos en blanco sobre fondo negro (como MNIST)
            cv2.line(img, (ix, iy), (x, y), (255, 255, 255), 20)
            ix, iy = x, y
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False

# Crear un canvas negro
img = np.zeros((400, 400, 3), np.uint8)
cv2.namedWindow('Dibuja un digito')
cv2.setMouseCallback('Dibuja un digito', draw_circle)

print("Instrucciones: Dibuja con el mouse. Presiona 'p' para predecir, 'c' para borrar, 'q' para salir.")

while True:
    cv2.imshow('Dibuja un digito', img)
    k = cv2.waitKey(1) & 0xFF

    if k == ord('q'): # Salir
        break
    elif k == ord('c'): # Limpiar pantalla
        img = np.zeros((400, 400, 3), np.uint8)
    elif k == ord('p'): # Predecir
        # 1. Convertir a escala de grises
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # 2. Redimensionar a 28x28 (lo que espera el modelo)
        resized = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)
        
        # 3. Normalizar y dar forma (batch_size, width, height, channels)
        input_data = resized / 255.0
        input_data = input_data.reshape(1, 28, 28, 1)
        
        # 4. Predicción
        prediction = model.predict(input_data)
        print("Prediction : ", prediction)
        digit = np.argmax(prediction)
        confidence = np.max(prediction) * 100
        
        print(f"Predicción: {digit} (Confianza: {confidence:.2f}%)")
        
        # Mostrar resultado en la imagen
        cv2.putText(img, f"Es un: {digit}", (10, 380), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

cv2.destroyAllWindows()

Instrucciones: Dibuja con el mouse. Presiona 'p' para predecir, 'c' para borrar, 'q' para salir.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
Prediction :  [[4.7808855e-05 9.9756795e-01 1.6231442e-04 7.7016557e-05 7.4703261e-05
  1.3873607e-03 5.0347968e-04 2.4068226e-05 1.3585332e-04 1.9487958e-05]]
Predicción: 1 (Confianza: 99.76%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Prediction :  [[9.2914604e-05 4.5809634e-06 9.9940205e-01 2.1985488e-06 7.4043751e-06
  5.5008509e-09 6.6659574e-11 4.8847677e-04 2.1518294e-07 2.1293140e-06]]
Predicción: 2 (Confianza: 99.94%)
